# Проект 1. Аналитическая записка: маржа сервиса ПРАЙМ

**Курс:** «Программирование на Python для бизнес-аналитики» · **Автор:** gvdernov

Я — аналитик и руководитель сервиса подписки **ПРАЙМ**. Основной продукт — платная подписка (списания
каждый месяц). Кроме того, подписчики получают **кэшбэк** за покупки в партнёрских сервисах — это деньги,
которые сервис отдаёт обратно.

Я отвечаю за вопрос : **растёт ли на самом деле маржа**
(выручка минус кэшбэк), и что двигает её изменениями. Эта записка — проверка гипотезы о кэшбэке
и факторная декомпозиция самого заметного провала маржи — августа 2026 года.

**Итог одной строкой:** кэшбэк — стабильная доля ≈4 % выручки и рост не съедает; августовское падение
маржи (−4,1 млн ₽) почти целиком объясняется снижением числа платежей на подписку — массовыми отменами
канала `partner_telecom` 22–23 августа, а не ценой и не кэшбэком.

## 1. Бизнес-вопрос и гипотезы

**Бизнес-вопрос**

> Подписка приносит выручку, но часть денег мы возвращаем кэшбэком за покупки в партнёрских сервисах.
> **Растёт ли маржа ПРАЙМ, и за счёт чего она меняется? Не съедает ли кэшбэк рост выручки?**
> И, раз в августе 2026 маржа заметно упала, — **за счёт чего** именно?

Это проверяемый вопрос: у нас есть ежемесячные списания (`payments`) и начисления кэшбэка (`cashback`),
и маржу можно посчитать по месяцам и разложить на факторы.

**Гипотезы**

| # | Гипотеза | Что увидим в данных |
|---|---|---|
| H0 | Кэшбэк растёт быстрее выручки, его доля увеличивается, маржа отстаёт от выручки | доля кэшбэка в выручке растёт от месяца к месяцу |
| H1 | Доля кэшбэка стабильна (~4 %), маржа растёт вместе с выручкой | доля кэшбэка колеблется около 4 %, маржа повторяет динамику выручки |
| H2 | Августовское падение вызвано оттоком, а не кэшбэком и не ценой | разложение июля→августа: главный фактор — число платежей на подписку (отмены), цена и кэшбэк — нет |

**План проверки:** 1) метрики и годовой тренд (H0 против H1); 2) «где» — аддитивная модель по каналам
и тарифам; 3) «за счёт чего» — мультипликативная модель и цепные подстановки (H2); 4) устойчивость
к длине месяца, обрезке августа и датам начисления кэшбэка; 5) вывод.

## 2. Зерно, период и конвенции расчёта

**Зерно.** Маржу считаем **за месяц** — это денежный поток отчётного месяца (списания идут раз в 30 дней,
начисления кэшбэка — по транзакциям месяца). Одна строка помесячного среза — пара **(месяц, канал
привлечения владельца, тариф)**. Такой уровень детализации достаточно крупный, чтобы вопрос про сервис
в целом не утонул в шуме отдельных клиентов, и достаточно мелкий, чтобы разрезать маржу по каналам
и тарифам.

- **Выручка** — сумма успешных списаний (`status = 'success'`): это деньги, которые реально пришли.
- Платёж приписываем **владельцу подписки** (`owner_client_id`): кто платит — тот и «строка». Участники
  шеринга выручки не создают, каждый платёж принадлежит ровно одной подписке и одному владельцу —
  **двойного счёта нет** (проверяем числом ниже).
- **Кэшбэк** — сумма начислений по дате начисления (`accrued_at`): конвенция курса (считали так же
  на семинаре про кэшбэк/LTV). Отдельно сверяем с датой покупки (`occurred_at` связанной транзакции) —
  вывод от выбора конвенции не зависит.
- **Период.** Для тренда берём полные месяцы с февраля 2025 по август 2026. Ключевое сравнение —
  **июль → август 2026** (последний полный месяц, в нём падение маржи).

**Оговорки о данных** (найдены при приёмке, проверяем кодом):

- январь 2025 неполный: история начинается 6 января — в тренде помечаем, в выводах не используем;
- **14–15 апреля 2026** нет успешных списаний — дыра в данных (апрель в ключевых выводах не участвует);
- **август 2026 обрезан**: последний день с данными — 30 августа (нет 31-го). Поэтому сравнение
  «июль→август» дублируем на **равных 30-дневных окнах** (1–30 июля против 1–30 августа) и в пересчёте
  на день — иначе «падение» августа перепутается с артефактом отсутствующего дня.

In [ ]:
import math

import matplotlib.pyplot as plt
import pandas as pd
from sqlalchemy import text

from prime.config import get_engine
import datetime

engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Сутки и месяцы — по UTC."""
    try:
        with engine.connect() as conn:
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


print("подключились как:", q("select current_user as login").iloc[0, 0])
print("pandas", pd.__version__)

In [ ]:
def load_monthly(client_filter: str = "true",
                 first: str = "2025-01-01", last: str = "2026-08-31") -> pd.DataFrame:
    """Месяц × тариф × канал: активные подписки, успешные платежи, выручка."""
    return q(f"""
        with seg as (
            select client_id, acquisition_channel as channel
            from prime.clients
            where {client_filter}
        ),
        act as (
            select to_char(m.month_start, 'YYYY-MM') as month, s.tariff_id, c.channel, count(*) as active
            from generate_series(cast(:first as timestamptz), cast(:last as timestamptz), interval '1 month') as m(month_start)
            join prime.subscriptions s
              on s.started_at < m.month_start + interval '1 month'
             and (s.ended_at is null or s.ended_at >= m.month_start)
            join seg c on c.client_id = s.owner_client_id
            group by 1, 2, 3
        ),
        rev as (
            select to_char(p.paid_at, 'YYYY-MM') as month, s.tariff_id, c.channel,
                   count(*) as n_payments, sum(p.amount) as revenue
            from prime.payments p
            join prime.subscriptions s on s.subscription_id = p.subscription_id
            join seg c on c.client_id = s.owner_client_id
            where p.status = 'success'
              and p.paid_at >= cast(:first as timestamptz)
              and p.paid_at <  cast(:last as timestamptz) + interval '1 month'
            group by 1, 2, 3
        )
        select a.month, a.tariff_id, a.channel, a.active,
               coalesce(r.n_payments, 0) as n_payments, coalesce(r.revenue, 0) as revenue
        from act a
        left join rev r
          on r.month = a.month and r.tariff_id = a.tariff_id and r.channel is not distinct from a.channel
        order by 1, 2, 3
    """, first=first, last=last)


monthly = load_monthly()
monthly["channel"] = monthly["channel"].fillna("без канала")
print(len(monthly), "строк (месяц × тариф × канал)")

# Контроль двойного счёта: каждый успешный платёж — одна строка, один владелец.
control = q("""
    select count(*) as n_pay,
           count(distinct p.payment_id) as n_distinct_pay,
           count(distinct s.subscription_id) as n_sub
    from prime.payments p
    join prime.subscriptions s on s.subscription_id = p.subscription_id
    where p.status = 'success'
""")
assert control.iloc[0]["n_pay"] == control.iloc[0]["n_distinct_pay"], "платёж задвоен"
print("контроль: платежей", f"{control.iloc[0]['n_pay']:,}".replace(",", " "),
      "— все разные, по", f"{control.iloc[0]['n_sub']:,}".replace(",", " "), "подпискам")
print("      => шеринг двойного счёта не создаёт: у подписки один владелец-плательщик")

In [ ]:
FIRST, LAST = "2025-01-01", "2026-08-31"

# Кэшбэк по месяцам начисления
cashback_m = q(f"""
    select to_char(accrued_at, 'YYYY-MM') as month,
           count(*) as n_cashback, sum(bonus_amount) as amount
    from prime.cashback
    where accrued_at >= cast(:first as timestamptz)
      and accrued_at <  cast(:last as timestamptz) + interval '1 month'
    group by 1 order by 1
""", first=FIRST, last=LAST)

# Кэшбэк по месяцам ПОКУПКИ (для сверки конвенции дат)
cashback_txn_m = q(f"""
    select to_char(t.occurred_at, 'YYYY-MM') as txn_month,
           count(*) as n_cashback, sum(b.bonus_amount) as amount
    from prime.cashback b
    join prime.transactions t on t.transaction_id = b.transaction_id
    where t.occurred_at >= cast(:first as timestamptz)
      and t.occurred_at <  cast(:last as timestamptz) + interval '1 month'
    group by 1 order by 1
""", first=FIRST, last=LAST)

# Кэшбэк по месяцам × канал привлечения
cashback_ch_m = q(f"""
    select to_char(b.accrued_at, 'YYYY-MM') as month, c.acquisition_channel as channel,
           count(*) as n_cashback, sum(b.bonus_amount) as amount
    from prime.cashback b
    join prime.transactions t on t.transaction_id = b.transaction_id
    join prime.clients c on c.client_id = t.client_id
    where b.accrued_at >= cast(:first as timestamptz)
      and b.accrued_at <  cast(:last as timestamptz) + interval '1 month'
    group by 1, 2 order by 1, 2
""", first=FIRST, last=LAST)

# Посуточные ряды: дни с успешными списаниями и дни с начислениями кэшбэка
daily = q(f"""
    select paid_at::date as day, count(*) as n, sum(amount) as revenue
    from prime.payments
    where status = 'success'
      and paid_at >= cast(:first as timestamptz)
      and paid_at <  cast(:last as timestamptz) + interval '1 month'
    group by 1 order by 1
""", first=FIRST, last=LAST)
daily["day"] = pd.to_datetime(daily["day"])

cashback_daily = q(f"""
    select accrued_at::date as day, sum(bonus_amount) as amount
    from prime.cashback
    where accrued_at >= cast(:first as timestamptz)
      and accrued_at <  cast(:last as timestamptz) + interval '1 month'
    group by 1 order by 1
""", first=FIRST, last=LAST)
cashback_daily["day"] = pd.to_datetime(cashback_daily["day"])

print("кэшбэк по месяцам:", len(cashback_m), "строк; дней со списаниями:", len(daily))

## 3. Метрики: выручка, кэшбэк, маржа по месяцам

Маржа месяца = выручка от подписок (успешные списания) **минус** начисленный кэшбэк.

В таблице ниже **одна строка — один месяц**: активные подписки, число успешных платежей, выручка,
кэшбэк, маржа и вспомогательные факторы — выручка и маржа на подписку, платежей на подписку, средняя
цена платежа. Дополнительно считаем выручку и маржу в пересчёте на «день с данными» — чтобы месяцы
разной длины и дыры в данных не путались с падениями бизнеса (дни считаются по фактическому наличию
списаний: в августе их 30, в апреле — 28 из-за дыры 14–15-го).

In [ ]:
rev_m = monthly.groupby("month").agg(active=("active", "sum"),
                                     n_payments=("n_payments", "sum"),
                                     revenue=("revenue", "sum"))
cb = cashback_m.set_index("month")["amount"].rename("cashback")
m = rev_m.join(cb).fillna(0)
m = m[m.index <= "2026-08"]                      # кэшбэк с сентябрьским хвостом не нужен

m["margin"] = m["revenue"] - m["cashback"]
m["cb_share"] = 100 * m["cashback"] / m["revenue"]

# дни, в которые реально были успешные списания (в августе их 30, в апреле 28 и т.п.)
days_data = daily.assign(month=daily["day"].dt.strftime("%Y-%m")).groupby("month")["day"].nunique()
m["days_data"] = days_data
m["rev_pday"] = m["revenue"] / m["days_data"]
m["margin_pday"] = m["margin"] / m["days_data"]

# факторы для декомпозиции
m["rev_per_sub"] = m["revenue"] / m["active"]
m["pay_per_sub"] = m["n_payments"] / m["active"]
m["price"] = m["revenue"] / m["n_payments"]
m["cb_per_sub"] = m["cashback"] / m["active"]
m["margin_per_sub"] = m["margin"] / m["active"]

pd.set_option("display.float_format", lambda x: f"{x:,.0f}".replace(",", " "))
m.round(0)

**График 1.** Динамика выручки, маржи и кэшбэка по месяцам (вся база).

Вывод, который доказывает график: **маржа идёт в ногу с выручкой** (кривые почти параллельны),
а кэшбэк — маленькая (3–4 %) и ровная полоса снизу. Скачок марта 2026 — подорожание тарифов
1 марта (+25 % Базовый, +12,5 % Расширенный), плюс февраль короче марта. Январь-2025 неполный
(данные с 6.01), апрель-2026 — с дырой 14–15 числа: оба помечены.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8), layout="constrained")

m[["revenue", "margin"]].plot.line(ax=ax, color=["#9CA3AF", "#374151"], linewidth=2)
m["cashback"].plot.line(ax=ax, color="#C1272D", linewidth=2)

ax.axvline("2026-03", color="#E5E7EB", linewidth=2)
ax.annotate("подорожание 01.03.2026", xy=("2026-03", 5.0e7), xytext=("2026-03", 7.0e7),
            ha="left", color="#6B7280", fontsize=9)
for bad, note in [("2025-01", "неполный"), ("2026-04", "дыра 14–15.04")]:
    ax.axvline(bad, color="#E5E7EB", linewidth=1.5, linestyle=":")
    ax.annotate(note, xy=(bad, 0.35e7), xytext=(bad, 0.8e7), ha="left", color="#9CA3AF", fontsize=8)

ax.set_ylabel("₽ в месяц")
ax.set_xlabel("месяц")
ax.set_title("Маржа растёт вместе с выручкой, кэшбэк — стабильная малая доля (≈4 %)",
             loc="left", fontsize=13)
ax.legend(["выручка", "маржа", "кэшбэк"], frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#E5E7EB")
ax.tick_params(axis="x", labelrotation=45)
plt.show()

In [ ]:
# Доля кэшбэка в выручке: вся база + персональный сегмент (клиенты с транзакциями
# в приложении, апрель 2026). Если строку сегмента получить нельзя — считаем только базу.
try:
    CLIENT_FILTER = q("select client_filter from prime.assignments where login = current_user").iloc[0, 0]
    SEGMENT_OK = True
except Exception:
    CLIENT_FILTER, SEGMENT_OK = None, False

if SEGMENT_OK:
    seg_monthly = load_monthly(CLIENT_FILTER)
    seg_cb = q(f"""
        select to_char(b.accrued_at, 'YYYY-MM') as month, sum(b.bonus_amount) as amount
        from prime.cashback b
        join prime.transactions t on t.transaction_id = b.transaction_id
        join prime.clients c on c.client_id = t.client_id
        where c.client_id in (select client_id from prime.clients where {CLIENT_FILTER})
          and b.accrued_at >= cast(:first as timestamptz)
          and b.accrued_at <  cast(:last as timestamptz) + interval '1 month'
        group by 1 order by 1
    """, first=FIRST, last=LAST)
    seg_rev = seg_monthly.groupby("month")["revenue"].sum()
    seg_share = 100 * seg_cb.set_index("month")["amount"] / seg_rev
    seg_share = seg_share[seg_share.index <= "2026-08"]
    print(f"доля кэшбэка в сегменте, {len(seg_share)} мес.; средняя за 2026: {seg_share.loc['2026-01':].mean():.1f}%")
else:
    print("сегмент не загружен — считаем только всю базу")

**График 2.** Доля кэшбэка в выручке по месяцам — вся база и мой сегмент
(активные пользователи приложения).

Проверка H0 против H1: если бы кэшбэк «съедал» рост, его доля ползла бы вверх. **Она стабильна** —
около **4 %** по всей базе в 2026 году (пунктир — среднее). У моего сегмента доля почти вдвое выше
(~7–8 %): активные пользователи приложения транзакций больше — но и там доля не растёт. Гипотеза
**H0 не подтверждается**.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.2), layout="constrained")

if SEGMENT_OK:
    s = seg_share.loc["2026-01":"2026-08"]
    ax.plot(s.index, s.values, color="#C1272D", linewidth=2, label="мой сегмент (приложение)")

base_share = m.loc["2026-01":"2026-08", "cb_share"]
ax.plot(base_share.index, base_share.values, color="#6B7280", linewidth=2, label="вся база")
mean_base = base_share.mean()
ax.axhline(mean_base, color="#9CA3AF", linewidth=1, linestyle="--")
ax.annotate(f"среднее ≈{mean_base:.1f}%", xy=("2026-08", mean_base), xytext=(0, 7),
            textcoords="offset points", ha="right", va="baseline",
            color="#6B7280", fontsize=9)
# запас сверху, чтобы подпись среднего не упиралась в край фигуры (и без сегмента)
ax.set_ylim(top=max(ax.get_ylim()[1], mean_base + 2.5))


ax.set_ylabel("% выручки")
ax.set_xlabel("месяц")
ax.set_title("Доля кэшбэка стабильна ≈4% — кэшбэк не растёт быстрее выручки", loc="left", fontsize=13)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#E5E7EB")
ax.tick_params(axis="x", labelrotation=45)
plt.show()

## 4. Факторная декомпозиция: июль → август 2026

Сначала «где» — аддитивная модель: маржа — сумма маржей каналов, значит её изменение — сумма изменений
каналов (правило курса: сумма влияний обязана сойтись с изменением итога, проверяем `assert`-ом).
Затем «за счёт чего» — мультипликативная модель и цепные подстановки.

### Шаг 1. «Где»: изменение маржи по каналам привлечения

In [ ]:
# Маржа канала = выручка канала − кэшбэк клиентов канала (в месяц)
rev_ch = monthly.groupby(["month", "channel"], dropna=False)[["active", "n_payments", "revenue"]].sum().reset_index()
cb_ch = cashback_ch_m.groupby(["month", "channel"], dropna=False, as_index=False)[["amount"]].sum()
cb_ch = cb_ch.rename(columns={"amount": "cashback"})
cb_ch["channel"] = cb_ch["channel"].fillna("без канала")

margin_ch = rev_ch.merge(cb_ch, on=["month", "channel"], how="left")
margin_ch["cashback"] = margin_ch["cashback"].fillna(0.0)
margin_ch["margin"] = margin_ch["revenue"] - margin_ch["cashback"]

jul_aug = margin_ch[margin_ch["month"].isin(["2026-07", "2026-08"])]
piv = jul_aug.pivot_table(index="channel", columns="month", values="margin", aggfunc="sum")
deltas = piv["2026-08"] - piv["2026-07"]

# Сумма влияний каналов обязана сойтись с изменением итоговой маржи
total_delta = m.loc["2026-08", "margin"] - m.loc["2026-07", "margin"]
assert abs(deltas.sum() - total_delta) < 1.0, "каналы не сходятся с итогом"

table = pd.DataFrame({"июль": piv["2026-07"], "август": piv["2026-08"], "изменение": deltas})
table["доля падения, %"] = (100 * deltas / total_delta).round(0)
print(table.round(0).to_string())
print(f"итого изменение маржи: {total_delta:,.0f} ₽".replace(",", " "))

**Вывод «где»:** упали **все** каналы — дело не в одном случайном канале. Но больше половины падения
дал **`partner_telecom` (≈56 %)**. Дальше проверяем гипотезу о причине: это отток, а не цена.

По тарифам картина симметрична (Базовый и Расширенный падают одинаково), тариф провал не объясняет —
останавливаемся на каналах.

In [ ]:
# Срез по тарифам для полноты: выручка и активность каждого тарифа в июле-августе.
# Кэшбэк сюда не мешаем — из декомпозиции видно, что на величину провала он не влияет.
rev_t = monthly.groupby(["month", "tariff_id"])[["active", "revenue"]].sum().reset_index()
tariffs = rev_t[rev_t["month"].isin(["2026-07", "2026-08"])].pivot_table(
    index="tariff_id", columns="month", values=["active", "revenue"], aggfunc="sum")
print("Активные подписки и выручка по тарифам, июль/август 2026:")
print(tariffs.round(0).to_string())
print()
print("Оба тарифа падают симметрично => тариф не объясняет провал, работаем с каналами.")

**График 3.** Каскад изменения маржи июль→август по каналам (тыс. ₽). Август обрезан 31-м числом —
что из падения «реальное», а что календарь с отсутствующим днём, проверяем на равных 30-дневных окнах
(раздел 5).

Заголовок графика — вывод: **«56 % падения маржи — канал partner_telecom»**. Красный — отрицательный
вклад, зелёный — положительный, серый — итог.

In [ ]:
def waterfall(deltas: pd.Series, title: str) -> plt.Axes:
    """Каскад: по столбику на фактор (вклад от нуля) и «итого» в конце. Ось включает ноль."""
    colors = ["#C1272D" if v < 0 else "#2A9D8F" for v in deltas]

    fig, ax = plt.subplots(figsize=(9, 4.6), layout="constrained")
    bars = ax.bar(deltas.index, deltas, bottom=0, color=colors, zorder=2)
    total = ax.bar("итого", deltas.sum(), color="#6B7280", zorder=2)
    for group in (bars, total):
        ax.bar_label(group, fmt=lambda v: f"{v:+,.0f}".replace(",", " ").replace("-", "−"), padding=3)
    ax.axhline(0, color="#111827", linewidth=0.8)
    ax.set_title(title, loc="left", fontsize=13)
    ax.set_ylabel("тыс. ₽")
    ax.margins(y=0.15)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="#E5E7EB")
    return ax


ax = waterfall(deltas.sort_values() / 1000, "56 % падения маржи — канал partner_telecom")
ax.set_xlabel("июль → август 2026. Каналы по убыванию падения.")
plt.show()

### Что произошло: массовые отмены 22–23 августа

Прежде чем множить факторы — посмотрим на подписки. Август 2026 выбивается из здоровой картины:
отток подписок (доля закончившихся из активных на начало месяца) обычно держится около 4 %,
а в августе — скачок. Основной вклад — массовые отмены **22–23 августа**.

In [ ]:
# Отток по месяцам: активные на начало месяца и закончившиеся за месяц (вся база)
churn = q(f"""
    with months as (
        select generate_series(cast(:first as timestamptz), cast(:last as timestamptz), interval '1 month') as month_start
    ),
    act as (
        select m.month_start, s.subscription_id,
               s.ended_at is not null and s.ended_at < m.month_start + interval '1 month' as ended_in_month
        from months m
        join prime.subscriptions s
          on s.started_at < m.month_start
         and (s.ended_at is null or s.ended_at >= m.month_start)
    )
    select to_char(month_start, 'YYYY-MM') as month,
           count(*) as active_at_start,
           count(*) filter (where ended_in_month) as churned
    from act
    group by 1 order by 1
""", first=FIRST, last=LAST)
churn = churn[(churn["month"] >= "2026-05")]
churn["churn_rate"] = 100 * churn["churned"] / churn["active_at_start"]
print("Отток по месяцам (2026-05 → 2026-08):")
print(churn.round(2).to_string(index=False))

# А теперь «обвал»: отмены 20–26 августа по каналам
spike = q("""
    select s.ended_at::date as day, c.acquisition_channel as channel, count(*) as n
    from prime.subscriptions s
    join prime.clients c on c.client_id = s.owner_client_id
    where s.ended_at >= '2026-08-20' and s.ended_at < '2026-08-27'
    group by 1, 2 order by 1, 2
""")
by_day = spike.pivot_table(index="day", columns="channel", values="n", aggfunc="sum").fillna(0)
by_day["всего"] = by_day.sum(axis=1)
top_channels = by_day[by_day.index < datetime.date(2026, 8, 31)].drop(columns="всего").sum().sort_values(ascending=False).head(3).index
print()
print("Отмены 20–26 августа по дням (топ-3 канала и итог):")
print(by_day[list(top_channels) + ["всего"]].astype(int).to_string())

### Шаг 2. «За счёт чего»: маржа = активные подписки × маржа на подписку

Мультипликативная модель (как на семинаре по разложению выручки):

> маржа = активные подписки × маржа на одну активную подписку

где маржа на подписку = платежей на подписку × цена платежа **−** кэшбэк на подписку.

Порядок подстановок: **сначала количество (активные), потом цена** — и внутри: сначала число платежей,
потом цена платежа, потом кэшбэк. Сумма влияний обязана сойтись с изменением маржи (−4,12 млн ₽).

In [ ]:
def contrib(x0: dict, x1: dict, order: list[str]) -> dict:
    """Влияние каждого фактора: меняем факторы по одному в порядке order."""
    cur, out = dict(x0), {}
    for f in order:
        before = math.prod(cur.values())
        cur[f] = x1[f]
        out[f] = math.prod(cur.values()) - before
    total = math.prod(x1.values()) - math.prod(x0.values())
    assert abs(sum(out.values()) - total) < 0.01, "сумма влияний не равна изменению"
    return out


# --- маржа = активные × маржа_на_подписку ---
n0, mps0 = m.loc["2026-07", "active"], m.loc["2026-07", "margin_per_sub"]
n1, mps1 = m.loc["2026-08", "active"], m.loc["2026-08", "margin_per_sub"]

by_active = (n1 - n0) * mps0          # сначала количество, при старой марже на подписку
by_mps = n1 * (mps1 - mps0)          # потом маржа на подписку при новом количестве
delta_margin = m.loc["2026-08", "margin"] - m.loc["2026-07", "margin"]
assert abs(by_active + by_mps - delta_margin) < 1.0
print(f"активные подписки:  {by_active:>+14,.0f} ₽".replace(",", " "))
print(f"маржа на подписку:  {by_mps:>+14,.0f} ₽".replace(",", " "))
print(f"сумма:               {by_active + by_mps:>+14,.0f} ₽   (изменение маржи {delta_margin:,.0f} ₽)".replace(",", " "))

# --- разворачиваем «маржа на подписку»: платежей × цена − кэшбэк (на подписку) ---
pp0, pr0 = m.loc["2026-07", "pay_per_sub"], m.loc["2026-07", "price"]
pp1, pr1 = m.loc["2026-08", "pay_per_sub"], m.loc["2026-08", "price"]
cp0, cp1 = m.loc["2026-07", "cb_per_sub"], m.loc["2026-08", "cb_per_sub"]

by_pay = (pp1 - pp0) * pr0            # сначала количество платежей, при старой цене
by_price = pp1 * (pr1 - pr0)          # потом цена при новом количестве платежей
assert abs(by_pay + by_price - (m.loc["2026-08", "rev_per_sub"] - m.loc["2026-07", "rev_per_sub"])) < 1e-6
by_cb = -(cp1 - cp0) * n1             # рост кэшбэка на подписку уменьшает маржу

print()
print(f"платежей на подписку: {by_pay * n1:>+14,.0f} ₽".replace(",", " "))
print(f"цена платежа:         {by_price * n1:>+14,.0f} ₽".replace(",", " "))
print(f"кэшбэк на подписку:   {by_cb:>+14,.0f} ₽".replace(",", " "))

factors = pd.Series({"активные подписки": by_active,
                     "платежей на подписку": by_pay * n1,
                     "цена платежа": by_price * n1,
                     "кэшбэк на подписку": by_cb})
assert abs(factors.sum() - delta_margin) < 1.0, "сумма влияний не равна изменению маржи"
print()
print("итого по факторам, тыс. ₽:", (factors / 1000).round(0).to_dict())

**График 4.** Каскад влияний факторов на изменение маржи июль→август (тыс. ₽).

Два числа в заголовке — суть: **цена платежа почти не изменилась (−46 тыс. ₽), кэшбэк — и вовсе −2 тыс. ₽**.
Падение дало одно: **меньше платежей на одну активную подписку** (0,977 → 0,901 за месяц: и длина месяца
тут не виновата, см. раздел 5). Это и есть след отмен 22–23 августа — часть подписок в августе уже
не списывалась. Гипотеза **H2 подтверждается**: дело в оттоке, не в цене и не в кэшбэке.

In [ ]:
ax = waterfall(factors.sort_values() / 1000,
               "Цена и кэшбэк ни при чём: падение маржи — это меньше платежей на подписку")
ax.set_xlabel("июль → август 2026. Порядок подстановок: активные → платежи → цена → кэшбэк.")
plt.show()

## 5. Устойчивость вывода

Три возражения, которые обязан снять разбор:

1. **Длина месяца.** Август короче июля, и в августе обрываются данные 31-го числа. Проверяем
   **равные 30-дневные окна** (1–30 июля против 1–30 августа) и **маржа в день**.
2. **Кэшбэк по месяцам начисления vs по месяцам покупки.** Если бы конвенция меняла вывод — расчёт
   был бы хрупким. Сверяем обе.
3. **Остаток от декомпозиции** — совместный член цепных подстановок: он мал, вывод не зависит
   от порядка подстановок.

In [ ]:
# --- 1) Равные 30-дневные окна по посуточным рядам ---
def window_sum(frame, col, a, b):
    return frame[(frame["day"] >= a) & (frame["day"] < b)][col].sum()

r_jul = window_sum(daily, "revenue", pd.Timestamp("2026-07-01"), pd.Timestamp("2026-07-31"))
r_aug = window_sum(daily, "revenue", pd.Timestamp("2026-08-01"), pd.Timestamp("2026-08-31"))
c_jul = window_sum(cashback_daily, "amount", pd.Timestamp("2026-07-01"), pd.Timestamp("2026-07-31"))
c_aug = window_sum(cashback_daily, "amount", pd.Timestamp("2026-08-01"), pd.Timestamp("2026-08-31"))

rows = pd.DataFrame({
    "окно 1–30 число": ["июль", "август", "изменение"],
    "выручка, ₽": [r_jul, r_aug, r_aug - r_jul],
    "маржа, ₽": [r_jul - c_jul, r_aug - c_aug, (r_aug - c_aug) - (r_jul - c_jul)],
}).set_index("окно 1–30 число")
print("Равные 30-дневные окна (июль без 31-го, август по факту):")
print(rows.round(0).to_string())

# --- 2) Выручка и маржа в день (по дням с данными) ---
print()
print("Маржа и выручка в день, тыс. ₽:")
print(m.loc["2026-05":"2026-08", ["rev_pday", "margin_pday"]].div(1000).round(1).to_string())

# --- 3) Конвенция дат кэшбэка: начисление vs покупка ---
cb_acc = cashback_m.set_index("month")["amount"]
cb_txn = cashback_txn_m.set_index("txn_month")["amount"].rename("cb_by_txn")
conv = pd.DataFrame({"по начислению": cb_acc, "по покупке": cb_txn}).fillna(0)
conv["разница, ₽"] = conv["по начислению"] - conv["по покупке"]
conv["разница, % от выручки"] = 100 * conv["разница, ₽"] / m["revenue"]
print()
print("Кэшбэк за 2026: конвенция начисления против покупки")
print(conv.loc["2026-07":"2026-08"].round(0).to_string())
print("Максимальная разница конвенций по всем месяцам 2026, % выручки:",
      f"{conv.loc['2025-02':'2026-08', 'разница, % от выручки'].abs().max():.2f}")

**Что показала проверка устойчивости:**

- **Равные 30-дневные окна:** падение маржи не «съедается» календарём — оно сокращается с −4,12 млн ₽
  (полные месяцы) примерно до **−2,11 млн ₽** (окно 1–30). То есть заметная часть «падения» — это
  отсутствующий день данных + разная длина месяцев, но **реальное падение остаётся**.
- **В пересчёте на день** маржа проседает даже на равных окнах: 1 966 551 → 1 894 623 ₽ в день (−3,6 %).
- **Конвенция кэшбэка** (начисление vs покупка) почти не меняет числа — максимум четверть процента
  выручки. Выбор конвенции на вывод не влияет.
- **Совместный член** цепных подстановок на этом переходе мал (тысячи рублей против миллионов),
  поэтому порядок подстановок ответ не искажает.

**График 5.** Маржа и выручка в день по месяцам (₽ на день с данными), с февраля 2025.

График отвечает на «не артефакт ли это»: до августа выручка и маржа в день **росли ровно** почти каждый
месяц (с поправкой на дыры), а в августе **обе проседают**. Значит у падения одна причина —
не календарь, а бизнес-событие (массовые отмены).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.4), layout="constrained")

view = m.loc["2025-02":"2026-08"]
ax.plot(view.index, view["rev_pday"].values / 1e6, color="#9CA3AF", linewidth=2, label="выручка в день")
ax.plot(view.index, view["margin_pday"].values / 1e6, color="#C1272D", linewidth=2, label="маржа в день")
ax.scatter(["2026-08"], [view.loc["2026-08", "margin_pday"] / 1e6], color="#C1272D", zorder=3)

ax.set_ylabel("млн ₽ в день")
ax.set_xlabel("месяц")
ax.set_title("В пересчёте на день маржа до августа росла, в августе — просела", loc="left", fontsize=13)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#E5E7EB")
ax.tick_params(axis="x", labelrotation=45)
plt.show()

## 6. Вывод за 30 секунд

**Что случилось.** Маржа ПРАЙМ (выручка от подписок минус кэшбэк) в августе 2026 упала на
**≈4,1 млн ₽** к июлю — с 60,96 до 56,84 млн ₽. Даже на равных 30-дневных окнах (август обрезан
31-м числом) падение ≈2,1 млн ₽, а маржа в день снизилась ≈на 3,6 %. До августа маржа в пересчёте
на день росла почти каждый месяц.

**За счёт чего.** Разложение `маржа = активные × маржа на подписку` (июль→август):
активные подписки выросли (+2,9 тыс., **+0,89 млн ₽**), но маржа на подписку упала 305 → 280 ₽
(**−5,0 млн ₽**). Внутри неё: **меньше платежей на подписку** (0,977 → 0,901; −4,96 млн ₽), а цена
платежа (−0,05 млн ₽) и кэшбэк (−0,002 млн ₽) **ни при чём**. «Где»: 56 % падения дал канал
`partner_telecom` — 22–23 августа отменены десятки тысяч подписок (отток августа 17 % против обычных 4 %).

**Чего мы не знаем.** Точное разделение «длина месяца + обрезка дня» и «эффект отмен» декомпозиция
даёт приблизительно (равные окна показывают порядок эффекта, не точность). Мы не знаем, разовая ли
эта отмена партнёра или начало новой нормы: сентябрьских данных ещё нет. Кэшбэк за августовские
покупки частично начислен уже в сентябре и в августовскую маржу не вошёл — на порядок влияния это
не влияет (доли процента).

**Что делать.**
1. **Канальная устойчивость:** 56 % падения — один канал. Разобрать контракт с телеком-партнёром
   на предмет массовых отмен (предложить win-back и проверить, не поменялись ли условия пакета).
   Диверсифицировать привлечение, чтобы ни один канал не мог уронить маржу на миллионы за два дня.
2. **Кэшбэк не трогать:** гипотеза «кэшбэк съедает рост» не подтвердилась — доля стабильна ≈4 %
   (в моём сегменте ~7–8 %, но тоже без роста). Повышать ставки кэшбэка сейчас нет оснований;
   добавить ежемесячный мониторинг доли кэшбэка как индикатор «проедания» маржи.
3. **Мониторинг оттока по каналам:** внедрить алерт на аномальный отток (как 22–23 августа), чтобы
   следующая потеря канала была видна в день события, а не в конце месяца.